[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/40_adam.ipynb)

# 🟡 Medium: Adam Optimizer

*Training*
Implement the **Adam** update as a pure function over pytrees.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \qquad
  v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t} \qquad
  \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

$$\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

### Signature
```python
def adam_update(params, grads, state, step, lr=1e-3,
                b1=0.9, b2=0.999, eps=1e-8):
    # state: {"m": pytree_like_params, "v": pytree_like_params}
    # step:  1-based — the first call has step=1
    # returns (new_params, new_state)
```

### Rules
- Do **not** use `optax`
- `params`, `grads`, `m` and `v` are all the same pytree structure — use
  `jax.tree.map`, do not assume a flat array
- `state` starts as all-zeros `m` and `v`
- Must work under `jax.jit`

### What bias correction actually fixes
This is the part people get wrong. $m$ and $v$ start at **zero**, so at $t=1$,
$m_1 = (1-\beta_1) g_1 = 0.1 g_1$ — a tenth of the true gradient. Without
correction the first steps are far too small, and because $\beta_2 = 0.999$ the
second-moment estimate takes *thousands* of steps to warm up.

The correction has a sharp observable signature: **with** it, the very first
update has magnitude $\approx \eta$ regardless of how large or small the
gradient is (since $\hat{m}_1/\sqrt{\hat{v}_1} = g/|g| = \pm 1$). That property
is exactly what the tests check, and it is the cleanest way to tell a correct
Adam from one missing the correction.

### Where AdamW differs
AdamW does **not** add weight decay to the gradient. It applies
$\theta \mathrel{-}= \eta \lambda \theta$ separately, so the decay is not scaled
by $\sqrt{\hat{v}}$. Folding L2 into the gradient instead — plain "Adam + L2" —
decays large-gradient parameters *less*, which is why AdamW generalises better
and is the default for transformers.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def adam_update(params, grads, state, step, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
    """One Adam step.

    Args:
        params: pytree of parameters
        grads:  pytree of gradients, same structure as params
        state:  {"m": pytree, "v": pytree} — both zeros on the first step
        step:   1-based step counter (first call is 1)

    Returns:
        (new_params, new_state)
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

params = {"w": jnp.array([1.0, -2.0])}
state = {"m": jax.tree.map(jnp.zeros_like, params),
         "v": jax.tree.map(jnp.zeros_like, params)}

# Wildly different gradient magnitudes...
for g in (jnp.array([1e-4, 1e-4]), jnp.array([1e4, 1e4])):
    p, _ = adam_update(params, {"w": g}, state, step=1, lr=0.1)
    print(f"grad {g[0]:>8.0e} -> step size {abs(float(p['w'][0] - 1.0)):.4f}")
# ...both give a first step of ~lr. That is bias correction doing its job.

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("adam")

# hint("adam")      # stuck? nudge without the answer
# solution("adam")  # spoiler: the reference implementation